# NBA API Connection

Pulls stats from NBA.com via `nba_api` (no key needed). Results are cached to `data/` so re-runs are instant and don't hammer NBA.com.

The 2026-27 season hasn't started yet, so `SEASON` defaults to last season. Change it once games begin.

In [1]:
import time
from pathlib import Path

import pandas as pd
from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import leaguedashplayerstats, playergamelog

SEASON = "2025-26"

# data/ folder in the project root, whether run from notebooks/ or the root
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 50)

## 1. Offline lookups (no network call)

In [2]:
print(len(players.get_active_players()), "active players,", len(teams.get_teams()), "teams")
players.find_players_by_full_name("Jokic")

530 active players, 30 teams


[{'id': 203999,
  'full_name': 'Nikola Jokić',
  'first_name': 'Nikola',
  'last_name': 'Jokić',
  'is_active': True}]

## 2. Season per-game averages for every player (cached)

In [3]:
def season_stats(season=SEASON, refresh=False):
    """Per-game stats for all players in a season, cached to data/."""
    path = DATA / f"player_stats_{season}.csv"
    if path.exists() and not refresh:
        return pd.read_csv(path)
    df = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season, per_mode_detailed="PerGame", timeout=60
    ).get_data_frames()[0]
    df.to_csv(path, index=False)
    return df

stats = season_stats()

# The 9 standard fantasy categories
cols = ["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "MIN",
        "PTS", "REB", "AST", "STL", "BLK", "FG3M", "FG_PCT", "FT_PCT", "TOV"]
stats[cols].sort_values("PTS", ascending=False).head(15)

,PLAYER_NAME,TEAM_ABBREVIATION,GP,MIN,PTS,REB,AST,STL,BLK,FG3M,FG_PCT,FT_PCT,TOV
372,Luka Dončić,LAL,64,35.8,33.5,7.7,8.3,1.6,0.5,4.0,0.476,0.780,4.0
501,Shai Gilgeous-Alexander,OKC,68,33.2,31.1,4.3,6.6,1.4,0.8,1.7,0.553,0.879,2.2
31,Anthony Edwards,MIN,61,35.0,28.8,5.0,3.7,1.4,0.8,3.4,0.489,0.796,2.9
260,Jaylen Brown,BOS,71,34.4,28.7,6.9,5.1,1.0,0.4,2.0,0.477,0.795,3.6
554,Tyrese Maxey,PHI,70,38.0,28.3,4.1,6.6,1.9,0.8,3.1,0.462,0.892,2.4
154,Donovan Mitchell,CLE,70,33.5,27.9,4.5,5.7,1.5,0.3,3.2,0.483,0.865,2.8
320,Kawhi Leonard,LAC,65,32.1,27.9,6.4,3.6,1.9,0.4,2.6,0.505,0.892,2.0
429,Nikola Jokić,DEN,65,34.8,27.7,12.9,10.7,1.4,0.8,1.7,0.569,0.831,3.7
186,Giannis Antetokounmpo,MIL,36,28.9,27.6,9.8,5.4,0.9,0.7,0.4,0.624,0.650,3.2
279,Joel Embiid,PHI,38,31.6,26.9,7.7,3.9,0.6,1.2,1.4,0.489,0.854,2.9


## 3. One player's game log

In [4]:
def game_log(name, season=SEASON):
    matches = players.find_players_by_full_name(name)
    if not matches:
        raise ValueError(f"No player found for {name!r}")
    time.sleep(0.6)  # be polite to NBA.com between requests
    return playergamelog.PlayerGameLog(
        player_id=matches[0]["id"], season=season, timeout=60
    ).get_data_frames()[0]

log = game_log("Nikola Jokic")
log[["GAME_DATE", "MATCHUP", "MIN", "PTS", "REB", "AST", "STL", "BLK", "FG3M", "TOV"]].head(10)

,GAME_DATE,MATCHUP,MIN,PTS,REB,AST,STL,BLK,FG3M,TOV
0,"Apr 12, 2026",DEN @ SAS,18,23,8,1,0,1,1,0
1,"Apr 08, 2026",DEN vs. MEM,31,14,16,10,2,0,0,2
2,"Apr 06, 2026",DEN vs. POR,43,35,14,13,5,2,1,6
3,"Apr 04, 2026",DEN vs. SAS,44,40,8,13,0,3,1,0
4,"Apr 01, 2026",DEN @ UTA,33,15,17,12,0,0,0,5
5,"Mar 29, 2026",DEN vs. GSW,36,25,15,8,1,0,4,2
6,"Mar 27, 2026",DEN vs. UTA,39,33,15,12,1,0,2,7
7,"Mar 25, 2026",DEN vs. DAL,38,23,21,19,1,0,1,1
8,"Mar 24, 2026",DEN @ PHX,35,23,17,17,0,0,0,2
9,"Mar 22, 2026",DEN vs. POR,35,22,14,14,2,1,2,5
